In [ ]:
# 3A (ECG). Implement ECG anomaly detection for the given ECG dataset using Autoencoder
# and build the model by using the following steps:
# • Dataset: Each row is one ECG signal of 140 time-steps; the last column is the label
#   (1 = Normal, 0 = Abnormal).
# • Goal: Identify abnormal ECG signals by training an autoencoder only on normal signals.
# Steps:
# • Preprocess the data by scaling features.
# • Train an autoencoder on normal ECG signals to learn typical heartbeat patterns.
# • Calculate reconstruction errors for signals and set a threshold to detect anomalies.
# • Compile the models with Optimizer, Loss, and Evaluation Metric

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [ ]:
data = pd.read_csv("/Users/aayushagarwal/DL/LP IV Updated Datasets/AutoEncoder/ECG/ecg.csv" , header=None)
print(data)

In [ ]:
X = data.iloc[: , :-1].values
y = data.iloc[: , -1].values

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [ ]:
x_train = x_train[y_train == 1]
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
class Autoencoder(keras.Model):
    def __init__(self , output_units):
        super().__init__()

        self.encoder = keras.Sequential([
            keras.layers.Dense(64 , activation = "relu"),
            keras.layers.Dropout(0.1),
            keras.layers.Dense(32 , activation = "relu"),
            keras.layers.Dropout(0.1),
            keras.layers.Dense(16 , activation = "relu"),
            keras.layers.Dropout(0.1),
            keras.layers.Dense(8 , activation = "relu"),
        ])

        self.decoder = keras.Sequential([
            keras.layers.Dense(16 , activation = "relu"),
            keras.layers.Dropout(0.1),
            keras.layers.Dense(32 , activation = "relu"),
            keras.layers.Dropout(0.1),
            keras.layers.Dense(64 , activation = "relu"),
            keras.layers.Dropout(0.1),
            keras.layers.Dense(output_units , activation = "linear"),
        ])

    def call(self , inputs):
        return self.decoder(self.encoder(inputs))

model = Autoencoder(x_train.shape[1])


In [ ]:
model.compile(optimizer="adam" , loss="mse")

In [ ]:
history = model.fit(x_train , x_train , epochs=30 , batch_size=64 , validation_data=(x_test , x_test))

In [ ]:
plt.plot(history.history["loss"] , label = "Traning Loss")
plt.plot(history.history["val_loss"] , label = "Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

In [ ]:
def get_errors(original , reconstructed):
    return np.mean(
        (original - reconstructed) ** 2 , axis = 1
    )

In [ ]:
train_errors = get_errors(x_train , model.predict(x_train , verbose=0))

In [ ]:
threshold = np.percentile(train_errors , 95)
print(threshold)

In [ ]:
test_errors = get_errors(x_test , model.predict(x_test , verbose = 0))
predictions = np.where(test_errors > threshold , 0 , 1)


In [ ]:
print(classification_report(y_test , predictions , target_names=["Abnormal" , "Normal"]))